# Modelo RNN (LSTM) — falha de porta em 7 dias

O Modelo 5 de Random Forest testava para ver se o histórico de alarmes de uma porta prevê uma falha nos próximos 7 dias, usando features feitas em janelas móveis (1/3/7/14/30 dias). Este notebook testa a mesma pergunta com uma rede neural recorrente (RNN), que recebe uma sequência não processada de alarmes e aprende sozinha o que importa olhar no tempo, sem ter alguém para desenhar as janelas na mão.

## 1. Comparação com o Modelo 5

Foi pensaod que para uma comparação entre dois modelos fazer sentido, os dois precisam responder à mesma pergunta com mesmo alvo e mesmas métricas. Assim a diferença pode ser atribuida às diferentes arquiteturas.

Este modelo usa a sequência de alarmes por porta específica, a mesma unidade de análise do Modelo 5 (porta-dia) e o mesmo alvo (`y = 1` se houve uma ordem corretiva emergencial, ZM01, nos 7 dias seguintes). Assim a comparação é feita com o Modelo 5, os dois preveem  a mesma coisa, só que com estruturas diferentes.

## 2. O que é uma Rede Neural Recorrente (RNN)

O Random Forest recebe uma linha por porta por dia com números já calculados; quantos alarmes teve nos últimos 7 dias, nos últimos 30, etc. Cada uma dessas colunas foi pensada antes de treinar o modelo.

Porém com uma RNN esse processo é diferente: ela recebe a sequência de dias, um de cada vez, na ordem em que aconteceram, e processa cada dia levando em conta o que ela "lembra" dos dias anteriores. Ao processar o dia 2, ela já sabe o que viu no dia 1; ao processar o dia 3, ela já sabe o que aconteceu nos dias 1 e 2, e assim por diante até o último dia da sequência (nesse caso, os 30 dias antes de `t0`). Esse resumo do que já viu até agora se chama hidden state, e é ele que carrega a memória de um dia pro outro.

No fim da sequência, o hidden state guarda um resumo dos 30 dias, aprendido pela própria rede durante o treino, e não desenhado à mão como as janelas móveis do Modelo 5. Uma última camada transforma esse resumo em uma probabilidade de falha.

## 3. Por que LSTM em vez de uma RNN simples

O problema de uma RNN simples é que, conforme a sequência cresce, a rede vai esquecendo o que viu muitos passos atrás, ou seja informação do dia 1 praticamente some do hidden state até chegar no dia 30.

**LSTM** (Long Short-Term Memory) resolve isso adicionando uma segunda memória, mais estável, que passa de um dia pro outro quase sem mudar, a não ser que a rede decida explicitamente atualizar ou apagar parte dela. Ela faz isso com três gates, que a própria rede aprende a usar durante o treino:

- **portão de esquecimento:** decide o que apagar da memória;
- **portão de entrada:** decide o que adicionar do dia atual;
- **portão de saída:** decide o que passar adiante pro próximo dia.

Essas decisões não são programadas, a rede aprende os pesos desses portões olhando para os dados de treino, do mesmo jeito que o Random Forest aprende onde cortar cada árvore olhando pros dados. Na prática, isso significa que a LSTM consegue notar, por exemplo, que uma porta teve um alarme importante há 3 semanas mesmo que os dias mais recentes estejam calmos, algo que uma RNN simples teria "esquecido".